# 01 - Explore NYC Yellow Taxi data

In [ ]:
import duckdb
con = duckdb.connect('../include/data/nyc_taxi.duckdb', read_only=True)
con.sql('DESCRIBE raw_yellow_trips')

In [ ]:
# Row counts per loaded file
con.sql('SELECT source_file, count(*) AS rows FROM raw_yellow_trips GROUP BY 1 ORDER BY 1')

In [ ]:
# Null counts per column
con.sql('SUMMARIZE raw_yellow_trips').df()[['column_name','column_type','null_percentage','min','max']]

In [ ]:
# Date range - look for pickups outside the file's month
con.sql('''
SELECT source_file,
       min(tpep_pickup_datetime) AS first_pickup,
       max(tpep_pickup_datetime) AS last_pickup
FROM raw_yellow_trips GROUP BY 1
''')

In [ ]:
# Trip distance distribution - zeros, negatives, extreme outliers
con.sql('''
SELECT count(*) FILTER (WHERE trip_distance <= 0)  AS zero_or_negative,
       count(*) FILTER (WHERE trip_distance > 100)  AS over_100_miles,
       quantile_cont(trip_distance, [0.5, 0.95, 0.99]) AS p50_p95_p99,
       max(trip_distance) AS max_distance
FROM raw_yellow_trips
''')